<a href="https://colab.research.google.com/github/Sucharitha255/Gen-AI-Practise/blob/main/Experiment6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import sqlite3
from datetime import datetime
import re

# ----------------------------------------
# 1. Initialize persistent memory
# ----------------------------------------

DB_NAME = "learning_memory.db"

def initialize_database():
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS preferences (
                name TEXT PRIMARY KEY,
                value TEXT
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS conversations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                question TEXT,
                answer TEXT,
                created_at TEXT
            )
        """)


# ----------------------------------------
# 2. Store and retrieve preferences
# ----------------------------------------

def save_preference(name, value):
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            INSERT INTO preferences (name, value)
            VALUES (?, ?)
            ON CONFLICT(name)
            DO UPDATE SET value = excluded.value
        """, (name, value))


def get_preferences():
    with sqlite3.connect(DB_NAME) as conn:
        rows = conn.execute(
            "SELECT name, value FROM preferences"
        ).fetchall()

    return dict(rows)


# ----------------------------------------
# 3. Store conversation history
# ----------------------------------------

def save_conversation(question, answer):
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            INSERT INTO conversations
            (question, answer, created_at)
            VALUES (?, ?, ?)
        """, (
            question,
            answer,
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ))


def get_recent_history(limit=5):
    with sqlite3.connect(DB_NAME) as conn:
        rows = conn.execute("""
            SELECT question, answer, created_at
            FROM conversations
            ORDER BY id DESC
            LIMIT ?
        """, (limit,)).fetchall()

    return rows


# ----------------------------------------
# 4. Retrieve relevant past questions
# ----------------------------------------

def retrieve_relevant_memory(question):
    current_words = set(re.findall(
        r"\b[a-zA-Z]{3,}\b", question.lower()
    ))

    history = get_recent_history(50)
    matches = []

    for old_question, old_answer, _ in history:
        old_words = set(re.findall(
            r"\b[a-zA-Z]{3,}\b", old_question.lower()
        ))

        common_words = current_words & old_words

        if common_words:
            matches.append((
                len(common_words),
                old_question,
                old_answer
            ))

    # Most overlapping questions first
    matches.sort(key=lambda item: item[0], reverse=True)

    return matches[:3]


# ----------------------------------------
# 5. Generate a personalized explanation
# ----------------------------------------

def generate_answer(question):
    preferences = get_preferences()
    level = preferences.get("level", "beginner")
    style = preferences.get("style", "examples")
    subject = preferences.get("subject", "general learning")

    q = question.lower()

    # Small educational knowledge base
    concepts = {
        "loop": (
            "A loop repeats a block of code multiple times.",
            "for i in range(3):\n    print(i)",
            "This prints 0, 1, and 2."
        ),
        "variable": (
            "A variable is a name used to store a value.",
            "age = 20\nprint(age)",
            "The variable age stores the number 20."
        ),
        "array": (
            "An array stores multiple values in an ordered collection.",
            "numbers = [10, 20, 30]\nprint(numbers[0])",
            "The first element is 10. Python lists are used here "
            "as a simple array-like example."
        ),
        "function": (
            "A function is a reusable block of code.",
            "def greet():\n    print('Hello')\n\ngreet()",
            "Calling greet() executes the code inside the function."
        ),
        "recursion": (
            "Recursion occurs when a function calls itself.",
            "def count(n):\n    if n == 0:\n        return\n"
            "    print(n)\n    count(n - 1)\n\ncount(3)",
            "The function counts down until it reaches the base case."
        )
    }

    concept_name = next(
        (name for name in concepts if name in q), None
    )

    if concept_name:
        definition, example, explanation = concepts[concept_name]
    else:
        definition = (
            f"Let's learn about {question}. "
            "First, identify its meaning, main purpose, and "
            "one practical example."
        )
        example = ""
        explanation = (
            "This demo has a small built-in knowledge base. "
            "Add more concepts to improve its answers."
        )

    # Adapt the explanation using saved preferences
    answer = f"Learning level: {level}\n"
    answer += f"Subject preference: {subject}\n\n"
    answer += f"Simple explanation: {definition}\n"

    if style in ("examples", "step-by-step"):
        if example:
            answer += f"\nExample:\n{example}\n"
            answer += f"\nHow it works: {explanation}\n"

    if style == "step-by-step":
        answer += (
            "\nStudy steps:\n"
            "1. Understand the definition.\n"
            "2. Read the example.\n"
            "3. Run the code and change its values.\n"
        )

    if level == "advanced":
        answer += (
            "\nAdvanced study tip: Explore implementation details, "
            "edge cases, and time or space complexity where relevant."
        )
    elif level == "intermediate":
        answer += (
            "\nPractice tip: Modify the example and predict its output."
        )
    else:
        answer += (
            "\nBeginner tip: Start with the definition and run "
            "the example one line at a time."
        )

    # Use previous learning context
    memories = retrieve_relevant_memory(question)

    if memories:
        answer += "\n\nRelated past learning:\n"
        for score, old_question, old_answer in memories:
            answer += f"- You previously asked: {old_question}\n"

    return answer


# ----------------------------------------
# 6. Main adaptive learning agent
# ----------------------------------------

def learning_agent():
    initialize_database()

    print("\n=== PERSONALIZED LEARNING ASSISTANT ===")
    print("I can remember your learning preferences.")
    print("Commands: /prefs, /history, /quit")

    preferences = get_preferences()

    if not preferences:
        print("\nLet's personalize your learning experience.")

        level = input(
            "Your level (beginner/intermediate/advanced): "
        ).strip().lower()

        if level not in ("beginner", "intermediate", "advanced"):
            level = "beginner"

        style = input(
            "Style (simple/examples/step-by-step): "
        ).strip().lower()

        if style not in ("simple", "examples", "step-by-step"):
            style = "examples"

        subject = input(
            "Preferred subject (e.g., Python, DSA): "
        ).strip()

        save_preference("level", level)
        save_preference("style", style)
        save_preference("subject", subject or "general learning")

        print("\nPreferences saved!")
    else:
        print("\nWelcome back! Your preferences were retrieved.")
        print(preferences)

    while True:
        question = input(
            "\nWhat would you like to learn? "
        ).strip()

        if question.lower() == "/quit":
            print("Goodbye! Your memory is saved.")
            break

        elif question.lower() == "/prefs":
            print("Saved preferences:", get_preferences())
            continue

        elif question.lower() == "/history":
            for q, a, timestamp in get_recent_history():
                print(f"\n[{timestamp}] You: {q}")
                print(f"Assistant: {a[:250]}...")
            continue

        elif question.lower().startswith("my level is "):
            level = question[12:].strip().lower()

            if level in ("beginner", "intermediate", "advanced"):
                save_preference("level", level)
                print("Your learning level has been updated.")
            else:
                print("Choose beginner, intermediate, or advanced.")
            continue

        elif question.lower().startswith("my style is "):
            style = question[12:].strip().lower()

            if style in ("simple", "examples", "step-by-step"):
                save_preference("style", style)
                print("Your explanation style has been updated.")
            else:
                print("Choose simple, examples, or step-by-step.")
            continue

        if not question:
            continue

        answer = generate_answer(question)
        print("\n--- Personalized Answer ---")
        print(answer)

        save_conversation(question, answer)


if __name__ == "__main__":
    learning_agent()



=== PERSONALIZED LEARNING ASSISTANT ===
I can remember your learning preferences.
Commands: /prefs, /history, /quit

Welcome back! Your preferences were retrieved.
{'level': 'intermediate', 'style': 'examples', 'subject': 'DSA'}

What would you like to learn? DSA

--- Personalized Answer ---
Learning level: intermediate
Subject preference: DSA

Simple explanation: Let's learn about DSA. First, identify its meaning, main purpose, and one practical example.

Practice tip: Modify the example and predict its output.

What would you like to learn? Binarysearch

--- Personalized Answer ---
Learning level: intermediate
Subject preference: DSA

Simple explanation: Let's learn about Binarysearch. First, identify its meaning, main purpose, and one practical example.

Practice tip: Modify the example and predict its output.

Related past learning:
- You previously asked: Binarysearch


What would you like to learn? /quit
Goodbye! Your memory is saved.
